# F1 Pitstop — V6: Advanced Feature Eng + Ridge Meta
**Strategy:**
- **Feature Engineering** (interactions, advanced groupbys, target encoding)
- **5 base models** (CatBoost, XGBoost, LightGBM, HistGB, ExtraTrees)
- **Ridge Meta-learner** to eliminate overfitting
- Data cleaning to remove impossible outliers

In [ ]:
!pip install category_encoders

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

TARGET     = 'PitNextLap'
ID_COL     = 'id'
FOLDS      = 5
SEED       = 42

import glob
KAGGLE_DATA = '/kaggle/input'
if os.path.exists(KAGGLE_DATA):
    train_files = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if not train_files:
        print(f"Files in /kaggle/input:", os.listdir('/kaggle/input'))
        raise FileNotFoundError('train.csv not found in /kaggle/input')
    DATA_DIR = os.path.dirname(train_files[0])
else:
    DATA_DIR = '/Users/raghu/Downloads'

TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
SUB_PATH   = f'{DATA_DIR}/sample_submission.csv'

for p in [TRAIN_PATH, TEST_PATH, SUB_PATH]:
    assert os.path.exists(p), f'Missing: {p}'

print(f'Data from: {DATA_DIR}')

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train size before cleaning: {train_df.shape}")
# 1. Clean Data (Train only)
# Remove TyreLife == 0 and LapTime outliers
train_df = train_df[(train_df['TyreLife'] > 0) & (train_df['LapTime (s)'] < 200)].copy()
print(f"Train size after cleaning: {train_df.shape}")

all_data = pd.concat([train_df, test_df], ignore_index=True)
test_idx = all_data['PitNextLap'].isna()

# 2. Advanced Feature Engineering

# A. Interactions
all_data['Circuit_Driver'] = all_data['Race'].astype(str) + "_" + all_data['Driver'].astype(str)
all_data['Driver_Tyre'] = all_data['Driver'].astype(str) + "_" + all_data['Compound'].astype(str)
all_data['Circuit_Tyre'] = all_data['Race'].astype(str) + "_" + all_data['Compound'].astype(str)

# Convert to category codes
for col in ['Circuit_Driver', 'Driver_Tyre', 'Circuit_Tyre']:
    all_data[col] = all_data[col].astype('category').cat.codes

# B. GroupBy Aggregations
# Average LapTime per Driver per Race
driver_race_lap_stats = all_data.groupby(['Driver', 'Race'])['LapTime (s)'].agg(['mean', 'std']).reset_index()
driver_race_lap_stats.rename(columns={'mean': 'DR_LapTime_mean', 'std': 'DR_LapTime_std'}, inplace=True)
all_data = all_data.merge(driver_race_lap_stats, on=['Driver', 'Race'], how='left')

# Average TyreLife when pitting per Circuit
circuit_tyre_stats = all_data[all_data['PitNextLap'] == 1].groupby('Race')['TyreLife'].mean().reset_index()
circuit_tyre_stats.rename(columns={'TyreLife': 'Circuit_AvgPit_TyreLife'}, inplace=True)
all_data = all_data.merge(circuit_tyre_stats, on='Race', how='left')

# Fill NaNs with medians
for col in all_data.columns:
    if all_data[col].isna().any() and col not in [TARGET, ID_COL]:
        all_data[col] = all_data[col].fillna(all_data[col].median())

# Additional basic features
all_data['StintRatio'] = all_data['LapNumber'] / (all_data['TyreLife'] + 1)
all_data['IsSoft'] = (all_data['Compound'] == 'SOFT').astype(int)

X_train = all_data[~test_idx].drop(columns=[ID_COL, TARGET])
y_train = all_data[~test_idx][TARGET].astype(int)
X_test  = all_data[test_idx].drop(columns=[ID_COL, TARGET])
test_ids = all_data[test_idx][ID_COL]

print(f"Engineered features: {X_train.shape[1]}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer

import catboost as cb
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import Ridge
import category_encoders as ce

class ModelFactory:
    @staticmethod
    def get_models():
        return {
            'catboost': cb.CatBoostClassifier(iterations=800, depth=8, learning_rate=0.03, thread_count=-1, verbose=0, random_state=SEED),
            'xgboost': xgb.XGBClassifier(n_estimators=600, max_depth=7, learning_rate=0.03, tree_method='hist', n_jobs=-1, random_state=SEED),
            'lightgbm': lgb.LGBMClassifier(n_estimators=700, max_depth=8, learning_rate=0.03, n_jobs=-1, random_state=SEED, verbose=-1),
            'histgb': HistGradientBoostingClassifier(max_iter=500, max_depth=8, random_state=SEED),
            'extratrees': ExtraTreesClassifier(n_estimators=500, max_depth=12, n_jobs=-1, random_state=SEED)
        }

def kfold_target_encoding(X_tr, y_tr, X_val, X_te, cols):
    te = ce.TargetEncoder(cols=cols)
    X_tr_enc = te.fit_transform(X_tr, y_tr)
    X_val_enc = te.transform(X_val)
    X_te_enc = te.transform(X_te)
    return X_tr_enc, X_val_enc, X_te_enc

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
models = ModelFactory.get_models()

oof_preds = {name: np.zeros(len(X_train)) for name in models}
test_preds = {name: np.zeros(len(X_test)) for name in models}

# Convert categories to strings/ints so Tree models can handle them uniformly
cat_cols = ['Race', 'Driver', 'Compound', 'Circuit_Driver', 'Driver_Tyre', 'Circuit_Tyre']
for c in cat_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].astype(str)
        X_test[c] = X_test[c].astype(str)

for fold, (idx_tr, idx_va) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- FOLD {fold+1}/{FOLDS} ---")
    
    X_tr, y_tr = X_train.iloc[idx_tr].copy(), y_train.iloc[idx_tr].copy()
    X_va, y_va = X_train.iloc[idx_va].copy(), y_train.iloc[idx_va].copy()
    X_te = X_test.copy()
    
    # K-Fold Target Encoding inside the loop
    target_cols = ['Race', 'Driver', 'Circuit_Driver']
    X_tr, X_va, X_te = kfold_target_encoding(X_tr, y_tr, X_va, X_te, target_cols)
    
    # Tree features
    for col in cat_cols:
        if col not in target_cols:
            X_tr[col] = X_tr[col].astype('category').cat.codes
            X_va[col] = X_va[col].astype('category').cat.codes
            X_te[col] = X_te[col].astype('category').cat.codes

    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_tr, y_tr)
        oof_preds[name][idx_va] = model.predict_proba(X_va)[:, 1]
        test_preds[name] += model.predict_proba(X_te)[:, 1] / FOLDS
        
        fold_auc = roc_auc_score(y_va, oof_preds[name][idx_va])
        print(f"{name} Fold {fold+1} AUC: {fold_auc:.5f}")

In [ ]:
# Meta-Model: Ridge Regression
print("\n--- META MODEL (RIDGE) ---")

OOF_df = pd.DataFrame(oof_preds)
TEST_df = pd.DataFrame(test_preds)

meta_model = Ridge(alpha=1.0, random_state=SEED)
final_oof = np.zeros(len(y_train))
final_test = np.zeros(len(X_test))

for fold, (idx_tr, idx_va) in enumerate(skf.split(OOF_df, y_train)):
    meta_model.fit(OOF_df.iloc[idx_tr], y_train.iloc[idx_tr])
    final_oof[idx_va] = meta_model.predict(OOF_df.iloc[idx_va])
    final_test += meta_model.predict(TEST_df) / FOLDS
    
meta_auc = roc_auc_score(y_train, final_oof)
print(f"\nFinal Meta OOF AUC: {meta_auc:.5f}")
print(f"Base models weights: {meta_model.coef_}")

sub = pd.DataFrame({ID_COL: test_ids, TARGET: final_test})
sub.to_csv('submission.csv', index=False)
print("Saved submission.csv")